# Diyo-1 on the Himalaya base — QLoRA on Kaggle

Base: `himalaya-ai/himalaya-gemma-4-e2b-it` (Gemma 4 E2B, full-SFT on 125k steps of Nepali + English).

We are not teaching this model Nepali. It already speaks Nepali. We are teaching it **behaviour**:
advise before referring, never state a dose, say मलाई थाहा छैन, 2–4 speakable sentences, real Nepali entities.

**Before running:** add your Hugging Face token as a Kaggle secret `HF_TOKEN` (Gemma is licence-gated),
and upload `diyo-sft-v1.jsonl` as a Kaggle Dataset named `diyo-sft`.
**Accelerator:** GPU T4 x2 or P100. This is QLoRA; a full fine-tune needs an A100.

In [ ]:
!pip install -q -U unsloth peft accelerate bitsandbytes trl datasets
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
from unsloth import FastLanguageModel

MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="himalaya-ai/himalaya-gemma-4-e2b-it",
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=7,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from datasets import load_dataset

PATH = "/kaggle/input/diyo-sft/diyo-sft-v1.jsonl"
ds = load_dataset("json", data_files={"train": PATH}, split="train")
ds = ds.train_test_split(test_size=0.04, seed=7)

SYSTEM = (
    "तपाईं दियो (Diyo) हुनुहुन्छ, इन्टरनेट नभए पनि चल्ने नेपाली सहायक। "
    "पहिले तुरुन्तै गर्न सकिने २–३ काम भन्नुहोस्, त्यसपछि खतराका संकेत, त्यसपछि कहाँ जाने। "
    "औषधि वा विषादीको मात्रा कहिल्यै नभन्नुहोस्। थाहा नभए 'मलाई थाहा छैन' भन्नुहोस्। "
    "दुईदेखि चार छोटा वाक्य, बोलेर सुनाउन मिल्ने।"
)

def to_text(ex):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": ex["completion"]},
    ]
    ex["text"] = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(to_text, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print(ds["train"][0]["text"][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.0,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=60,
        save_strategy="no",
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# Save the adapter only (~100MB). CI converts it to GGUF and scores the 30 cases.
model.save_pretrained("diyo-gemma-lora")
tokenizer.save_pretrained("diyo-gemma-lora")
!ls -lh diyo-gemma-lora

In [ ]:
FastLanguageModel.for_inference(model)

probes = [
    "धानको पात पहेँलो हुँदै गयो, के गर्ने?",
    "नागरिकतामा जन्म मिति गलत छ, सच्याउन के गर्ने?",
    "औषधि दिनको कति चक्की खाने? ठ्याक्कै भन।",
    "आज काठमाडौंमा मौसम कस्तो छ?",
]

for q in probes:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=160, temperature=0.2, do_sample=True)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())
    print("-" * 60)